# TD 6 — La courbe de Beveridge

**Analyse des données — L3 Économie**

Deux relations fondent l'analyse du marché du travail : la **courbe de Beveridge** — emplois vacants contre chômage — et la **courbe de Phillips** — inflation contre chômage. Toutes deux ne se voient qu'en graphique.

Tiennent-elles ?

Données : Eurostat, chargées par son interface de programmation. Aucun fichier à télécharger.

> *Exécution → Tout exécuter* avant de démarrer.

## Partie 1 — Charger et aligner

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"

URLS = {
    "vacances":  BASE + "jvs_q_nace2/Q.SA.B-S.TOTAL.JVR.FR?format=SDMX-CSV",
    "chomage":   BASE + "une_rt_q/Q.SA.Y15-74.PC_ACT.T.FR?format=SDMX-CSV",
    "inflation": BASE + "prc_hicp_manr/M.RCH_A.CP00.FR?format=SDMX-CSV",
}

def serie(url):
    """Ne garde que la periode et la valeur, et trie par date."""
    d = pd.read_csv(url)[["TIME_PERIOD", "OBS_VALUE"]]
    return d.sort_values("TIME_PERIOD").reset_index(drop=True)

vac = serie(URLS["vacances"])
cho = serie(URLS["chomage"])
inf = serie(URLS["inflation"])

for nom, d in [("vacances", vac), ("chomage", cho), ("inflation", inf)]:
    print("%-10s %3d obs | %s -> %s" % (nom, len(d), d["TIME_PERIOD"].iloc[0], d["TIME_PERIOD"].iloc[-1]))

**Question 1.** Quelle période couvre chaque série ? Laquelle limite la courbe de Beveridge ?

*Votre réponse :*

In [ ]:
# L'inflation est mensuelle : 2024-01, 2024-02...
# Le chomage et les vacances sont trimestriels : 2024-Q1, 2024-Q2...
print(inf["TIME_PERIOD"].head(3).tolist())
print(cho["TIME_PERIOD"].head(3).tolist())

**Question 2.** Avant de croiser l'inflation avec le chômage, que faut-il lui faire ?

**Question 3.** Pour passer de douze valeurs mensuelles à quatre trimestrielles, moyenne ou somme ? Justifiez à partir de ce que mesure la variable.

*Vos réponses :*

In [ ]:
# TODO : convertir l'inflation mensuelle en trimestres
inf["trimestre"] = (pd.PeriodIndex(inf["TIME_PERIOD"], freq="M")
                      .asfreq("Q").astype(str).str.replace("Q", "-Q"))

inf_t = inf.groupby("trimestre")["OBS_VALUE"].________().reset_index()
inf_t.columns = ["TIME_PERIOD", "inflation"]
inf_t.head()

## Partie 2 — La courbe de Beveridge

In [ ]:
# On rapproche les deux series trimestrielles
bev = vac.merge(cho, on="TIME_PERIOD", suffixes=("_vac", "_cho"))
bev.columns = ["trimestre", "vacances", "chomage"]
bev["annee"] = bev["trimestre"].str[:4].astype(int)

print(len(bev), "trimestres communs |", bev["trimestre"].iloc[0], "->", bev["trimestre"].iloc[-1])
bev.head()

In [ ]:
# TODO : le nuage, chomage en abscisse
plt.scatter(bev["________"], bev["________"])
plt.xlabel("taux de chômage (%)")
plt.ylabel("taux d'emplois vacants (%)")
plt.show()

**Question 4.** Décrivez la forme obtenue en une phrase.

*Votre réponse :*

In [ ]:
# TODO : relier les points dans l'ordre chronologique
plt.________(bev["chomage"], bev["vacances"], marker="o", markersize=3)
plt.xlabel("taux de chômage (%)")
plt.ylabel("taux d'emplois vacants (%)")
plt.show()

**Question 5.** Qu'apparaît-il que le nuage seul ne montrait pas ?

*Votre réponse :*

In [ ]:
# TODO : trois periodes, trois couleurs
def periode(a):
    if a < 2020:   return "avant 2020"
    if a <= 2021:  return "2020-2021"
    return "après 2021"

bev["periode"] = bev["annee"].apply(________)

for nom, g in bev.groupby("periode"):
    plt.scatter(g["chomage"], g["vacances"], label=nom)
plt.xlabel("taux de chômage (%)")
plt.ylabel("taux d'emplois vacants (%)")
plt.legend()
plt.show()

**Question 6.** À quel moment la courbe se déplace-t-elle ?

**Question 7.** Un déplacement vers le haut et la droite signifie : à chômage donné, davantage d'emplois vacants. Qu'est-ce que cela dit du fonctionnement du marché du travail ?

*Vos réponses :*

## Partie 3 — La courbe de Phillips

In [ ]:
phi = inf_t.merge(cho, on="TIME_PERIOD")
phi.columns = ["trimestre", "inflation", "chomage"]
phi["annee"] = phi["trimestre"].str[:4].astype(int)

print(len(phi), "trimestres |", phi["trimestre"].iloc[0], "->", phi["trimestre"].iloc[-1])

In [ ]:
# TODO : inflation en ordonnee, chomage en abscisse
plt.scatter(phi["________"], phi["________"])
plt.xlabel("taux de chômage (%)")
plt.ylabel("inflation (%)")
plt.show()

**Question 8.** Voyez-vous une relation ?

*Votre réponse :*

In [ ]:
# TODO : choisissez une annee de coupure et comparez les deux sous-periodes
COUPURE = ________

for nom, g in phi.groupby(phi["annee"] < COUPURE):
    plt.scatter(g["chomage"], g["inflation"],
                label=f"avant {COUPURE}" if nom else f"{COUPURE} et après")
plt.xlabel("taux de chômage (%)")
plt.ylabel("inflation (%)")
plt.legend()
plt.show()

In [ ]:
# TODO : la correlation, sur l'ensemble puis par sous-periode
print("ensemble :", round(phi["chomage"].corr(phi["________"]), 3))
for nom, g in phi.groupby(phi["annee"] < COUPURE):
    print(nom, ":", round(g["chomage"].corr(g["inflation"]), 3), "| n =", len(g))

**Question 9.** La relation change-t-elle d'aspect selon la sous-période ?

**Question 10.** Que concluez-vous sur la stabilité de la relation ?

**Question 11.** Vos trois graphiques de la partie 2 portaient sur les mêmes données. Lequel publieriez-vous, et que devez-vous écrire à côté ?

*Vos réponses :*

## Pour ceux qui ont terminé

**Question 12.** Lissez les deux séries par une moyenne mobile sur quatre trimestres — `rolling(4).mean()` — et retracez la courbe de Beveridge.

**Question 13.** Ajoutez l'Allemagne : remplacez `FR` par `DE` dans les adresses. Les deux pays se déplacent-ils au même moment ?

**Question 14.** Le taux d'emplois vacants n'existe qu'à partir de 2011. Cherchez une série française plus longue à la DARES, et dites ce qu'elle changerait.

*Vos réponses :*

## Avant la séance suivante

Vérifiez que le notebook s'exécute de haut en bas. Retenez la règle de la séance 6 : toute figure temporelle doit dire **pourquoi cette période** — et vous venez de l'éprouver vous-mêmes.

---

Supports, données et corrigés : `stefaniamarcassa.github.io/analyse_des_donnees`